# Análisis de customers.avro

**Autor:** Daniel Guzmán  
**Fecha:** 2026-04-23  
**Entorno:** Databricks

In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, when

In [0]:
catalog = "workspace"
schema = "default"
volume = "customers_files_daniel"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

print(path_volume)

In [0]:
inicio = time.time()

df_avro = (
    spark.read
    .format("avro")
    .load(f"{path_volume}/customers.avro")
)

total_registros = df_avro.count()
fin = time.time()

print(f"Tiempo de lectura: {fin - inicio:.4f} segundos")
print(f"Registros: {total_registros}")
print(f"Columnas: {df_avro.columns}")

In [0]:
display(df_avro.limit(5))

In [0]:
print("Tipos de datos:")
for col_name, dtype in df_avro.dtypes:
    print(f"{col_name}: {dtype}")

In [0]:
print("Shape:")
print(f"Filas: {df_avro.count()}")
print(f"Columnas: {len(df_avro.columns)}")

In [0]:
df_avro.printSchema()

In [0]:
nulos_df = df_avro.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_avro.columns
])

display(nulos_df)

In [0]:
for c in df_avro.columns:
    print(f"{c}: {df_avro.select(c).distinct().count()} valores únicos")

In [0]:
display(df_avro.describe())

In [0]:
print(df_avro.columns)

In [0]:
clientes_por_pais = (
    df_avro.groupBy("Country")
    .count()
    .withColumnRenamed("count", "total_clientes")
    .orderBy(F.col("total_clientes").desc())
)

display(clientes_por_pais)

In [0]:
display(clientes_por_pais.limit(5))

In [0]:
empresas_distintas = df_avro.select("Company").distinct().count()
print(f"Empresas distintas: {empresas_distintas}")

In [0]:
nombres_frecuentes = (
    df_avro.withColumn(
        "Nombre Completo",
        F.concat_ws(" ", F.col("First Name"), F.col("Last Name"))
    )
    .groupBy("Nombre Completo")
    .count()
    .orderBy(F.col("count").desc())
)

display(nombres_frecuentes.limit(1))

In [0]:
clientes_sin_ciudad = df_avro.filter(
    F.col("City").isNull() | (F.trim(F.col("City")) == "")
).count()

print(f"Clientes sin ciudad registrada: {clientes_sin_ciudad}")

## Reflexión sobre el formato .avro

- Tiempo de lectura registrado: 2.6386 segundos
- Tamaño del archivo: 1.80 MB aprox.
- Fue relativamente sencillo de leer con PySpark en Databricks.
- Como ventaja, Avro incluye esquema y es muy útil para intercambio de datos y streaming. Como desventaja, no es legible para humanos y suele ser menos cómodo para exploración manual que CSV o JSON.
- Usaría Avro en pipelines de integración, eventos, mensajería y procesos donde el esquema embebido aporte consistencia.